In [1]:
from datasets import load_from_disk

In [2]:
from transformers import AutoModelForTokenClassification
from transformers import AutoTokenizer
from transformers import Trainer



model_path = "E:\\PROJECTS\\Privacy-Risk-Analyser\\models\\xlm-roberta-ner" 
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)
tokenizer = tokenizer.from_pretrained(model_path)


In [3]:
# Loading dataset
dataset_dict = load_from_disk("E:/PROJECTS/Privacy-Risk-Analyser/privacy_ner_dataset")

In [4]:
# Loading tokenizer and define label list
model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

label_list = dataset_dict["train"].features["labels"].feature.names
label_to_id = {l: i for i, l in enumerate(label_list)}
num_labels = len(label_list)

In [5]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",         # Ensure uniform length
        max_length=128                # Or 256 depending on your model input limit
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Mask token
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs
# Mapping dataset
tokenized_datasets = dataset_dict.map(tokenize_and_align_labels, batched=True)

In [6]:
# Test data
test_dataset = tokenized_datasets["test"]  


In [7]:
from datasets import Dataset


test_dataset = Dataset.from_list(test_dataset)


tokenized_test_dataset = test_dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [8]:
trainer = Trainer(model=model, tokenizer=tokenizer)
predictions = trainer.predict(tokenized_test_dataset)


pred_labels = predictions.predictions.argmax(-1)


C:\Users\Dell\AppData\Local\Temp\ipykernel_25640\2237999240.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, tokenizer=tokenizer)


In [9]:
pred_labels

array([[8, 8, 8, ..., 8, 8, 8],
       [8, 8, 8, ..., 8, 8, 8],
       [8, 2, 6, ..., 8, 8, 8],
       ...,
       [8, 8, 8, ..., 8, 8, 8],
       [8, 8, 8, ..., 8, 8, 8],
       [8, 8, 8, ..., 8, 8, 8]], dtype=int64)

In [10]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report


true_labels = predictions.label_ids


true_labels_flat = []
pred_labels_flat = []

for pred, true in zip(pred_labels, true_labels):
    for p, t in zip(pred, true):
        if t != -100:
            true_labels_flat.append(t)
            pred_labels_flat.append(p)


In [11]:
accuracy = accuracy_score(true_labels_flat, pred_labels_flat)
print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.9061


In [13]:
from sklearn.metrics import classification_report
id2label = {i: label for i, label in enumerate(label_list)}

# Flatten true and predicted labels
true_labels = []
predicted_labels = []

for i, label in enumerate(tokenized_test_dataset["labels"]):
    true_labels.extend([id2label[l] for l in label if l != -100])  # ignore special tokens
    predicted_labels.extend([id2label[pred] for j, pred in enumerate(pred_labels[i]) if tokenized_test_dataset["labels"][i][j] != -100])

# Classification report
print(classification_report(true_labels, predicted_labels))


              precision    recall  f1-score   support

   B-ADDRESS       0.00      0.00      0.00        10
     B-EMAIL       0.00      0.00      0.00        20
      B-NAME       0.00      0.00      0.00       120
     B-PHONE       0.00      0.00      0.00         9
   I-ADDRESS       0.06      0.14      0.08        36
     I-EMAIL       0.29      0.27      0.28       165
      I-NAME       0.29      0.28      0.29       237
     I-PHONE       0.05      0.05      0.05        39
           O       0.96      0.95      0.96      9375

    accuracy                           0.91     10011
   macro avg       0.18      0.19      0.18     10011
weighted avg       0.91      0.91      0.91     10011

